# Colab 01 - Setup and checks

This notebook prepares a Colab runtime for fine-tuning MMAudio from `/content/MMAudio` while keeping large artifacts in Google Drive under `/content/drive/MyDrive/MMAudio_Yuqi`.

Expected Drive layout after setup:

```text
MMAudio_Yuqi/
  data/
    mmaudio_dataset_stride8/
      train/   # training videos
      val/     # validation videos
      test/    # test/evaluation videos
    mmaudio_dataset_stride4/
  ext_weights/
  weights/
  latents/
    stride8/
    stride4/
  outputs/
  eval_outputs/
  logs/
  tsv/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, '/content/MMAudio')

from fine_tune.urban_paths import (
    AUDIO_SR,
    CLIP_SEQ_LEN,
    DEFAULT_STRIDE,
    DRIVE_ROOT,
    EVAL_DIR,
    EXT_WEIGHTS_DIR,
    LATENT_SEQ_LEN,
    LOG_DIR,
    MODE,
    MODEL,
    OUTPUT_DIR,
    REPO_ROOT,
    SYNC_SEQ_LEN,
    TEXT_LABEL,
    TEXT_SEQ_LEN,
    TSV_DIR,
    VIDEO_EXTENSIONS,
    WEIGHTS_DIR,
    ensure_drive_dirs,
    latent_dir_for_stride,
    split_video_dirs,
)

STRIDE = DEFAULT_STRIDE
SPLIT_VIDEO_DIRS = split_video_dirs(STRIDE)
LATENTS_ROOT = latent_dir_for_stride(STRIDE)
RUNS_ROOT = OUTPUT_DIR
EVAL_ROOT = EVAL_DIR

ensure_drive_dirs(STRIDE)
for split_dir in SPLIT_VIDEO_DIRS.values():
    split_dir.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_ROOT)
print('Drive root:', DRIVE_ROOT)
print('Stride:', STRIDE)
print('Mode/model:', MODE, MODEL)
print('Latents:', LATENTS_ROOT)
print('Outputs:', RUNS_ROOT)
print('Eval:', EVAL_ROOT)
print('Logs:', LOG_DIR)

In [ ]:
%cd /content/MMAudio

# Install the local package and the evaluation dependency used by train.py.
# Re-run this cell after reconnecting to a fresh Colab runtime.
!python -m pip install -q -e .
!python -m pip install -q git+https://github.com/hkchengrex/av-benchmark.git

In [ ]:
%cd /content/MMAudio

# Keep repo-local paths compatible with scripts that expect ./weights and ./ext_weights,
# while storing the actual large files in Google Drive.
def link_drive_dir(repo_name: str, drive_path: Path):
    repo_path = REPO_ROOT / repo_name
    drive_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        return
    if repo_path.exists():
        if repo_path.is_dir() and not any(repo_path.iterdir()):
            repo_path.rmdir()
        else:
            print(f'{repo_path} exists; linking missing files inside it instead.')
            return
    repo_path.symlink_to(drive_path, target_is_directory=True)
    print(f'Linked {repo_path} -> {drive_path}')


def link_drive_file(repo_rel: str, drive_path: Path):
    repo_path = REPO_ROOT / repo_rel
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        return
    if repo_path.exists():
        if not drive_path.exists():
            repo_path.replace(drive_path)
            print(f'Moved {repo_path} -> {drive_path}')
        else:
            repo_path.unlink()
    repo_path.symlink_to(drive_path)
    print(f'Linked {repo_path} -> {drive_path}')


link_drive_dir('weights', WEIGHTS_DIR)
link_drive_dir('ext_weights', EXT_WEIGHTS_DIR)

for rel, target in {
    'weights/mmaudio_small_16k.pth': WEIGHTS_DIR / 'mmaudio_small_16k.pth',
    'ext_weights/v1-16.pth': EXT_WEIGHTS_DIR / 'v1-16.pth',
    'ext_weights/synchformer_state_dict.pth': EXT_WEIGHTS_DIR / 'synchformer_state_dict.pth',
    'ext_weights/best_netG.pt': EXT_WEIGHTS_DIR / 'best_netG.pt',
    'ext_weights/empty_string.pth': EXT_WEIGHTS_DIR / 'empty_string.pth',
}.items():
    link_drive_file(rel, target)

In [ ]:
%cd /content/MMAudio

from huggingface_hub import hf_hub_download
import urllib.request

# These are the files required for 16 kHz small-model training and evaluation.
hf_files = [
    'ext_weights/v1-16.pth',
    'ext_weights/synchformer_state_dict.pth',
    'ext_weights/best_netG.pt',
    'weights/mmaudio_small_16k.pth',
]

for filename in hf_files:
    target = DRIVE_ROOT / filename
    if target.exists() and target.stat().st_size > 0:
        print(f'Exists: {target}')
        continue
    print(f'Downloading {filename}')
    hf_hub_download(
        repo_id='hkchengrex/MMAudio',
        filename=filename,
        local_dir=str(DRIVE_ROOT),
        local_dir_use_symlinks=False,
    )

empty_string_path = EXT_WEIGHTS_DIR / 'empty_string.pth'
if not empty_string_path.exists() or empty_string_path.stat().st_size == 0:
    print('Downloading ext_weights/empty_string.pth')
    urllib.request.urlretrieve(
        'https://github.com/hkchengrex/MMAudio/releases/download/v0.1/empty_string.pth',
        empty_string_path,
    )
else:
    print(f'Exists: {empty_string_path}')

In [ ]:
%cd /content/MMAudio

import json
import subprocess
import torch

print('Python package path check')
assert REPO_ROOT.exists(), f'Missing repo: {REPO_ROOT}'

from mmaudio.model.sequence_config import CONFIG_16K

assert CONFIG_16K.latent_seq_len == LATENT_SEQ_LEN, CONFIG_16K.latent_seq_len
assert CONFIG_16K.clip_seq_len == CLIP_SEQ_LEN, CONFIG_16K.clip_seq_len
assert CONFIG_16K.sync_seq_len == SYNC_SEQ_LEN, CONFIG_16K.sync_seq_len
assert CONFIG_16K.sampling_rate == AUDIO_SR, CONFIG_16K.sampling_rate

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

required = [
    EXT_WEIGHTS_DIR / 'v1-16.pth',
    EXT_WEIGHTS_DIR / 'synchformer_state_dict.pth',
    EXT_WEIGHTS_DIR / 'best_netG.pt',
    EXT_WEIGHTS_DIR / 'empty_string.pth',
    WEIGHTS_DIR / 'mmaudio_small_16k.pth',
]

missing = [str(p) for p in required if not p.exists()]
print(json.dumps({str(p): p.exists() for p in required}, indent=2))
if missing:
    raise FileNotFoundError('\n'.join(missing))

In [ ]:
from pathlib import Path


def count_videos(root: Path):
    if not root.exists():
        return 0
    return sum(1 for p in root.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)

for split, root in SPLIT_VIDEO_DIRS.items():
    print(f'{split:>5}: {count_videos(root)} videos in {root}')

print('\nPlace your videos in the split folders above, then run colab_02_extract_latents.ipynb.')